# Thiết kế Prompt Template cho RAG (RAG Prompt Engineering & JSON Formatting)

Thực hiện việc thiết kế cấu trúc prompt hoàn chỉnh cho mô hình **Llama-3.1-8B (1-LoRA)**.
Mục tiêu là tích hợp ngữ cảnh từ RAG và hướng dẫn mô hình đưa ra kết quả đánh giá theo đúng cấu trúc định dạng JSON mong muốn.

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Nạp các biến môi trường cấu hình cache
load_dotenv(os.path.abspath("../.env"))

sys.path.append(os.path.abspath("../src"))
from rag import rag_utils

## 1. Định nghĩa Prompt Template dành cho 1-LoRA

Prompt này yêu cầu mô hình đóng vai trò là một giám khảo IELTS, phân tích từng tiêu chí thành phần dựa trên các ví dụ tham khảo từ RAG, sau đó xuất kết quả duy nhất ở định dạng JSON.

In [ ]:
IELTS_EVAL_PROMPT_TEMPLATE = """You are a highly experienced IELTS writing examiner. Your goal is to provide a precise and consistent evaluation of an essay by following a structured reasoning process.

**CONTEXT (Reference Essays with Scores):**
{context}

**NEW ESSAY TO GRADE:**
{question}

**EVALUATION PROCESS (Think step-by-step):**
1. Task Response (TR) Analysis: Assess how well the 'NEW ESSAY' addresses the prompt. Compare its quality to the TR scores in the 'CONTEXT'.
2. Coherence and Cohesion (CC) Analysis: Assess structure, paragraphing, and linking. Compare to CC scores in the 'CONTEXT'.
3. Lexical Resource (LR) Analysis: Assess range and accuracy of vocabulary. Compare to LR scores in the 'CONTEXT'.
4. Grammatical Range and Accuracy (GRA) Analysis: Assess grammar range and accuracy. Compare to GRA scores in the 'CONTEXT'.

**FINAL OUTPUT FORMAT (Strict JSON):**
Your entire response MUST be a single valid JSON object containing exactly these fields. Do NOT include markdown code blocks or explanations outside JSON.
{{
  "Task_Response": {{
    "Band": <score>,
    "Comment": "<brief justification>"
  }},
  "Coherence_and_Cohesion": {{
    "Band": <score>,
    "Comment": "<brief justification>"
  }},
  "Lexical_Resource": {{
    "Band": <score>,
    "Mistakes": ["<mistake1>", "<mistake2>"],
    "Corrections": ["<correction1>", "<correction2>"],
    "Comment": "<brief justification>"
  }},
  "Grammatical_Range_and_Accuracy": {{
    "Band": <score>,
    "Mistakes": ["<mistake1>", "<mistake2>"],
    "Corrections": ["<correction1>", "<correction2>"],
    "Comment": "<brief justification>"
  }},
  "General_Feedback": "<constructive feedback>"
}}

JSON Response:
"""

## 2. Tạo Thử nghiệm Prompt Hoàn chỉnh với RAG

Tải DB, truy xuất 2 tài liệu tham khảo và điền các tham số vào template để quan sát prompt cuối cùng.

In [ ]:
# Load DB
VECTOR_DB_DIR = "../data/processed/chroma_db"
vectordb = rag_utils.load_vector_db(VECTOR_DB_DIR)

# Thông tin đầu vào từ học sinh
student_prompt = "Some people believe that school children should be required to do community service. Discuss both views."
student_essay = "Nowadays, community service is considered as a good way for teenagers to learn about society. In my essay I will discuss about both sides..."

# 1. Truy xuất RAG
retrieved_docs = rag_utils.retrieve_examples(vectordb, student_essay, k=2)
context_str = rag_utils.format_rag_context(retrieved_docs)

# 2. Sinh prompt mẫu
final_prompt = rag_utils.format_evaluation_prompt(
    prompt_template=IELTS_EVAL_PROMPT_TEMPLATE,
    context=context_str,
    essay_prompt=student_prompt,
    essay_text=student_essay
)

print("=== PROMPT HOÀN CHỈNH SẼ GỬI ĐẾN LLAMA-3.1-8B ===\n")
print(final_prompt)